# 01 · 환자 요약 방식 — 네 가지 실측 비교

환자 한 명은 박동 수백 개를 갖는다. 그것을 환자값 하나로 줄이는 방법이 여럿이고,
어느 것이 나은지는 논증이 아니라 측정으로 가른다.

| 방식 | 어떻게 |
|---|---|
| ① 중앙값 | 박동별로 특징을 재고 환자 내 중앙값 |
| ② 분위수 | 같은 값들의 p25 · p50 · p75 (특징 수 3배) |
| ③ 앙상블 | 박동 파형을 겹쳐 평균한 합성 파형에서 특징을 1회 |
| ④ 대표 박동 | 중앙 특징 벡터에 가장 가까운 **실제** 박동 하나 |

세 지표로 잰다 — **재현성**(홀·짝 박동 분할 일치도), **검출률**, **질환 신호**(유의 셀 수·효과 크기).

박동 단위로 그대로 회귀하면 같은 사람이 수백 번 세어져 p값이 부풀어 오른다(의사반복).
그래서 환자가 분석 단위라는 점은 어느 방식을 쓰든 공통이다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("02_logistic/01_summary_method")
print("산출물 →", rep.dir)

In [ ]:
res = rep.path("summary_method_compare.csv")
if res.exists():
    display(pd.read_csv(res))
else:
    print("비교 산출물이 없다 — 아래 절차를 실행한다")

## 결과

| 방식 | 특징 | 재현성 | 최저 | 검출률 | 유의 셀 | 유의율 | \|log OR\| | 질환 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| **① 중앙값** | 20 | **0.989** | **0.895** | 1.000 | 196 | **5.7%** | **0.318** | 47 |
| ② 분위수 | 60 | 0.988 | 0.866 | 1.000 | 536 | 5.2% | 0.317 | **51** |
| ③ 앙상블 | 20 | 0.967 | 0.790 | 0.996 | 124 | 3.6% | 0.260 | 32 |
| ④ 대표 박동 | 20 | 0.929 | 0.734 | 0.998 | 176 | 5.1% | 0.279 | 40 |

### ③이 탈락한 이유 — 평균이 봉우리를 지운다

박동마다 반사파가 나타나는 **시점이 조금씩 다르다.** 겹쳐 평균하면 봉우리가 완만한 언덕으로
뭉개져 `find_peaks`에 잡히지 않는다.

| 특징 | ① 박동별 | ③ 앙상블 | ④ 대표 박동 |
|---|---:|---:|---:|
| RI | 0.998 | **0.102** | 0.512 |
| dT | 0.998 | **0.102** | 0.512 |
| d_over_a | 0.996 | 0.413 | 0.823 |
| LVET | 1.000 | 0.993 | 0.998 |

위치가 안정적인 중복절흔은 멀쩡하다(0.993). 앙상블은 **박동간 위치가 흔들리는 특징을 지운다.**

### ④가 ①에 못 미친 이유 — 표본이 하나

대표 박동은 실제 파형이라 반사파가 살아 있다(RI 0.512, 앙상블의 5배). 그러나 395개 중 하나만
쓰므로 그 하나가 우연히 어느 박동이냐에 값이 좌우된다. 재현성이 0.929(최저 0.734)로 가장 낮다.

### 확정 — ① 박동별 특징의 중앙값

"중앙값이 뭉갠다"는 우려와 반대로, 박동 수백 개에서 계산된 값들의 중심을 잡으면 측정 잡음이
크게 줄어든다. 정보를 버리는 대신 정밀도를 얻는 교환인데 이 자료에서는 그 교환이 이득이었다.
②는 질환 5개(I49 · I60 · I67 · J98 · L89)를 더 찾지만 검정 수가 3배가 되므로 **보충 분석**으로 둔다.

## 재현성 재산출 — 홀·짝 박동 분할

In [ ]:
from scipy.stats import spearmanr

B = pd.read_csv(paths.interim("beat_features_v2.csv"),
                usecols=["subject"] + F.MORPH,
                dtype={**{f: "float32" for f in F.MORPH}, "subject": str})
B["_i"] = B.groupby("subject").cumcount()
odd = B[B._i % 2 == 1].groupby("subject")[F.MORPH].median()
even = B[B._i % 2 == 0].groupby("subject")[F.MORPH].median()

rows = []
for c in F.MORPH:
    m = odd[c].notna() & even[c].notna()
    if m.sum() > 200:
        rows.append((c, int(m.sum()), spearmanr(odd[c][m], even[c][m])[0]))
rel = pd.DataFrame(rows, columns=["feature", "n", "홀짝 상관"]).sort_values("홀짝 상관")
print(f"중앙값 {rel['홀짝 상관'].median():.3f} · 최저 {rel['홀짝 상관'].min():.3f}")
rep.table(rel, "split_half_reliability.csv", "홀·짝 박동 분할 재현성")
rel.head(10).round(3)

## 산출물

In [ ]:
rep.done("환자 요약 방식 네 가지 실측 비교")
rep.summary()